In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a program for a multi-agent flocking simulation (boids). The input consists of:
</p>
  <li>An array <code>agents</code> containing <code>N</code> agents, where <code>N</code> is the total number of agents</li>
  <li>Each agent occupies 4 consecutive 32-bit floating point numbers in the array: $[x, y, v_x, v_y]$, where:
      <ul>
          <li>$(x, y)$ represents the agent's position in 2D space</li>
          <li>$(v_x, v_y)$ represents the agent's velocity vector</li>
      </ul>
  </li>
  <li>The total array size is <code>4 * N</code> floats, with agent $i$'s data stored at indices <code>[4i, 4i+1, 4i+2, 4i+3]</code></li>
</ul>

<h2>Simulation Rules</h2>
<ol>
  <li>For each agent $i$, identify all neighbors $j$ (where $i \neq j$) within radius $r = 5.0$ using:
      $$
      \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2} < r
      $$
  </li>
  <li>Compute average velocity of neighboring agents:
      $$
      \vec{v}_{avg} = \begin{cases}
      \frac{1}{|N_i|} \sum_{j \in N_i} \vec{v}_j & \text{if } |N_i| > 0 \\
      \vec{v}_i & \text{if } |N_i| = 0
      \end{cases}
      $$
      where $N_i$ is the set of neighbors for agent $i$
  </li>
  <li>Update velocity:
      $$
      \vec{v}_{new} = \vec{v} + \alpha(\vec{v}_{avg} - \vec{v}), \text{ where } \alpha = 0.05
      $$
  </li>
  <li>Update position:
      $$
      \vec{p}_{new} = \vec{p} + \vec{v}_{new}
      $$
  </li>
</ol>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>agents_next</code> array</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input: N = 2
agents = [
  0.0, 0.0, 1.0, 0.0,    // Agent 0: [x, y, vx, vy]
  3.0, 4.0, 0.0, -1.0    // Agent 1: [x, y, vx, vy]
]

Output:
agents_next = [
  1.0, 0.0, 1.0, 0.0,    // Agent 0: [x, y, vx, vy]
  3.0, 3.0, 0.0, -1.0    // Agent 1: [x, y, vx, vy]
]
</pre>

<h2>Constraints</h2>
<ul>
<li>1 &le; <code>N</code> &le; 100,000</li>
<li>Each agent's position and velocity components are 32-bit floats</li>

  <li>Performance is measured with <code>N</code> = 10,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// agents, agents_next are device pointers
extern "C" void solve(const float* agents, float* agents_next, int N) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# agents, agents_next are tensors on the GPU
@cute.jit
def solve(agents: cute.Tensor, agents_next: cute.Tensor, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# agents is a tensor on the GPU
@jax.jit
def solve(agents: jax.Array, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    agents: UnsafePointer[Float32, MutExternalOrigin],
    agents_next: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# agents, agents_next are tensors on the GPU
def solve(agents: torch.Tensor, agents_next: torch.Tensor, N: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# agents, agents_next are tensors on the GPU
def solve(agents: torch.Tensor, agents_next: torch.Tensor, N: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Multi-Agent Simulation", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free"
        )

    def reference_impl(self, agents: torch.Tensor, agents_next: torch.Tensor, N: int):
        assert agents.shape == (4 * N,)
        assert agents_next.shape == (4 * N,)
        assert agents.dtype == agents_next.dtype
        assert agents.device == agents_next.device
        r = 5.0
        r2 = r * r
        alpha = 0.05
        agents_reshaped = agents.view(N, 4)
        agents_next_reshaped = agents_next.view(N, 4)
        positions = agents_reshaped[:, :2]
        velocities = agents_reshaped[:, 2:]
        diff = positions.unsqueeze(1) - positions.unsqueeze(0)
        dist_sq = (diff**2).sum(dim=2)
        dist_sq.fill_diagonal_(r2 + 1)
        neighbor_mask = dist_sq < r2
        sum_velocities = neighbor_mask.float() @ velocities
        neighbor_counts = neighbor_mask.sum(dim=1, keepdim=True)
        avg_velocities = torch.empty_like(velocities)
        nonzero_mask = neighbor_counts[:, 0] > 0
        avg_velocities[nonzero_mask] = sum_velocities[nonzero_mask] / neighbor_counts[nonzero_mask]
        avg_velocities[~nonzero_mask] = velocities[~nonzero_mask]
        new_velocities = velocities + alpha * (avg_velocities - velocities)
        new_positions = positions + new_velocities
        agents_next_reshaped[:] = torch.cat([new_positions, new_velocities], dim=1)
        agents_next.copy_(agents_next_reshaped.view(-1))

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "agents": (ctypes.POINTER(ctypes.c_float), "in"),
            "agents_next": (ctypes.POINTER(ctypes.c_float), "out"),
            "N": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        N = 2
        agents = torch.tensor([0.0, 0.0, 1.0, 0.0, 5.0, 0.0, 0.0, 1.0], device="cuda", dtype=dtype)
        agents_next = torch.empty(4 * N, device="cuda", dtype=dtype)
        return {
            "agents": agents,
            "agents_next": agents_next,
            "N": N,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        test_cases = []
        # basic_example
        agents = torch.tensor([0.0, 0.0, 1.0, 0.0, 3.0, 4.0, 0.0, -1.0], device="cuda", dtype=dtype)
        agents_next = torch.empty(8, device="cuda", dtype=dtype)
        test_cases.append({"agents": agents, "agents_next": agents_next, "N": 2})
        # single_agent
        agents = torch.tensor([10.0, 15.0, 1.0, -1.0], device="cuda", dtype=dtype)
        agents_next = torch.empty(4, device="cuda", dtype=dtype)
        test_cases.append({"agents": agents, "agents_next": agents_next, "N": 1})
        # two_agents_interacting
        agents = torch.tensor([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0], device="cuda", dtype=dtype)
        agents_next = torch.empty(8, device="cuda", dtype=dtype)
        test_cases.append({"agents": agents, "agents_next": agents_next, "N": 2})
        # four_agents
        agents = torch.tensor(
            [0.0, 0.0, 1.0, 0.0, 2.0, 2.0, 0.0, 1.0, 4.0, 4.0, -1.0, 0.0, 6.0, 6.0, 0.0, -1.0],
            device="cuda",
            dtype=dtype,
        )
        agents_next = torch.empty(16, device="cuda", dtype=dtype)
        test_cases.append({"agents": agents, "agents_next": agents_next, "N": 4})
        # boundary_distance
        agents = torch.tensor(
            [0.0, 0.0, 1.0, 1.0, 3.0, 4.0, -1.0, -1.0], device="cuda", dtype=dtype
        )
        agents_next = torch.empty(8, device="cuda", dtype=dtype)
        test_cases.append({"agents": agents, "agents_next": agents_next, "N": 2})
        # medium_simulation (random)
        agents = torch.empty(4096, device="cuda", dtype=dtype).uniform_(-100.0, 100.0)
        agents_next = torch.empty(4096, device="cuda", dtype=dtype)
        test_cases.append({"agents": agents, "agents_next": agents_next, "N": 1024})
        return test_cases

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        agents = torch.empty(40000, device="cuda", dtype=dtype).uniform_(-1000.0, 1000.0)
        agents_next = torch.empty(40000, device="cuda", dtype=dtype)
        return {
            "agents": agents,
            "agents_next": agents_next,
            "N": 10000,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
